# 4.10 Self-Organizing Maps

Self-Organizing Maps turn nearest-representative learning into an ordered grid of prototypes.

A point updates its best matching unit and nearby units, so neighboring cells learn nearby regions of the data space. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

RNG = np.random.default_rng(7)


def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs


def scaled(X):
    return StandardScaler().fit_transform(X)


def plot_xy(X):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=0).fit_transform(X)


def safe_silhouette(X, labels):
    if len(np.unique(labels)) < 2:
        return float("nan")
    if len(np.unique(labels)) >= len(labels):
        return float("nan")
    return float(silhouette_score(X, labels))

def som_train(X, grid_shape=(2, 2), sigma=1.0, lr=0.4, steps=300, seed=0):
    rng = np.random.default_rng(seed)
    rows, cols = grid_shape
    n_units = rows * cols
    idx = rng.choice(X.shape[0], size=n_units, replace=True)
    weights = X[idx].copy()
    grid = np.array([[r, c] for r in range(rows) for c in range(cols)], dtype=float)

    for step in range(steps):
        x = X[step % X.shape[0]]
        distances = np.linalg.norm(weights - x, axis=1)
        bmu = int(np.argmin(distances))
        frac = 1.0 - step / max(steps - 1, 1)
        sigma_t = max(0.15, sigma * frac)
        lr_t = lr * frac
        grid_dist = np.sum((grid - grid[bmu]) ** 2, axis=1)
        influence = np.exp(-grid_dist / (2.0 * sigma_t * sigma_t))
        weights = weights + lr_t * influence[:, None] * (x - weights)

    assignments = np.argmin(pairwise_distances(X, weights), axis=1)
    return assignments, weights, grid


## The concept, built once on D1

The lesson's fit cost is

$$J=\sum_{i=1}^{m}\min_k \lVert x_i-\mu_k\rVert_2^2$$

For $x=[1,2]$, the squared distances are $0.25$ to $[1,1.5]$ and $16.25$ to $[4.5,4]$. Four such nearest distances give $J=1.00$.

In [ ]:

x = np.array([1.0, 2.0])
mu1 = np.array([1.0, 1.5])
mu2 = np.array([4.5, 4.0])
d1 = float(np.sum((x - mu1) ** 2))
d2 = float(np.sum((x - mu2) ** 2))
J = 4.0 * d1

assert d1 == 0.25
assert d2 == 16.25
assert J == 1.00

print("near distance", d1)
print("far distance", d2)
print("toy J", J)


The reusable SOM below implements real best-matching-unit assignment and Gaussian neighborhood updates over a CPU-sized grid.

In [ ]:

X_d1 = scaled(cluster_ladder()[0][1])
labels_d1, weights_d1, grid_d1 = som_train(X_d1, grid_shape=(1, 3), sigma=1.0, lr=0.5, steps=120, seed=0)

print("D1 BMU labels", labels_d1.tolist())
print("D1 weights")
print(np.round(weights_d1, 3))


## The dataset ladder

We use the shared F2 clustering ladder exactly once in the setup cell: hand points, clean blobs, anisotropic overlap, real Iris, and real digits 0–3. The hidden labels are kept only for audit metrics such as ARI; the clustering methods never train on them.

In [ ]:

rungs = cluster_ladder()

for idx, (name, X, y_true, k) in enumerate(rungs, start=1):
    print(idx, name, "shape=", X.shape, "k=", k, "labels=", np.unique(y_true).tolist())
    print("sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same method across D1–D5

Each rung uses a grid with at least as many cells as target clusters. BMU cells are compacted into cluster labels for silhouette scoring.

In [ ]:

results = []
artifacts = []

for idx, (name, X, y_true, k) in enumerate(cluster_ladder(), start=1):
    Xs = scaled(X)
    grid_shape = (2, 2) if k <= 4 else (2, 3)
    bmu, weights, grid = som_train(Xs, grid_shape=grid_shape, sigma=1.2, lr=0.5, steps=500, seed=idx)
    active = {unit: label for label, unit in enumerate(np.unique(bmu))}
    labels = np.array([active[unit] for unit in bmu])
    score = safe_silhouette(Xs, labels)
    ari = adjusted_rand_score(y_true, labels)
    results.append({"rung": idx, "name": name, "score": score, "ari": ari, "active": len(active)})
    artifacts.append((Xs, labels, bmu, weights, grid))

for row in results:
    print(row["rung"], row["name"], "silhouette=", round(row["score"], 3), "ARI=", round(row["ari"], 3), "active_cells=", row["active"])


## Results visualization

The first figure colors points by BMU-derived clusters; the second plots the shared silhouette metric as the ladder gets harder.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(18, 3))

for ax, row, artifact in zip(axes, results, artifacts):
    Xs, labels, bmu, weights, grid = artifact
    coords = plot_xy(Xs)
    ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="tab10", s=18)
    ax.set_title(f"D{row['rung']} sil={row['score']:.2f}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

plt.figure(figsize=(6, 3))
plt.plot([r["rung"] for r in results], [r["score"] for r in results], marker="o")
plt.xticks([1, 2, 3, 4, 5])
plt.xlabel("ladder rung")
plt.ylabel("silhouette")
plt.title("SOM silhouette vs complexity")
plt.show()


## Pitfall on D5: grid size and initialization create fragile maps

A tiny grid or one seed may make digit organization look decisive when it is unstable. The fix is scaling plus repeated seeds and a grid that can represent the expected groups.

In [ ]:

name, X5, y5, k5 = cluster_ladder()[-1]
Xs5 = scaled(X5)

bmu_bad, weights_bad, grid_bad = som_train(Xs5, grid_shape=(1, 2), sigma=0.4, lr=0.5, steps=250, seed=0)
labels_bad = bmu_bad
bad_score = safe_silhouette(Xs5, labels_bad)

scores = []

for seed in [0, 1, 2, 3, 4]:
    bmu_fix, weights_fix, grid_fix = som_train(Xs5, grid_shape=(2, 2), sigma=1.2, lr=0.5, steps=500, seed=seed)
    labels_fix = bmu_fix
    score_fix = safe_silhouette(Xs5, labels_fix)
    scores.append(score_fix)

print("too-small grid", round(bad_score, 3))
print("scaled repeated seeds mean", round(float(np.mean(scores)), 3))
print("scaled repeated seeds std", round(float(np.std(scores)), 3))


## Evaluate it + Practice

- Metric: track silhouette from BMU-derived clusters on every rung and compare against a no-skill baseline such as random labels with the same number of clusters.
- Sanity check: D1 and D2 should be visibly easier than D5; if not, inspect scaling and assignments before trusting the curve.
- Ablation: turn off the neighborhood by making sigma tiny
- Failure signals: large seed-to-seed variation, many empty cells, or a map that uses grid position as false evidence
- Baseline: random BMU assignments or KMeans without the topology update

Practice prompts:
1. Change one hyperparameter by a small amount and explain whether the D5 score moves for a meaningful reason.

2. Add a random-label baseline to the results table and compare it with the method.

3. Pick one D5 point, print its assignment evidence, and decide whether the method is confident or ambiguous.